# Training YOLO11 on custom dataset

In [ ]:
!nvidia-smi

## All imports for Training

In [ ]:
import os
from pathlib import Path

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import ultralytics
from roboflow import Roboflow
from ultralytics import YOLO, settings

HOME = Path(os.getcwd())
print(HOME)

## Install YOLO11 via Ultralytics

In [ ]:
settings.reset()

ultralytics.checks()

In [ ]:
ultralytics.settings["tensorboard"] = True
ultralytics.settings["weights_dir"]     = str(HOME / "weights")
ultralytics.settings["runs_dir"]        = str(HOME / "runs")
ultralytics.settings["datasets_dir"]    = str(HOME / "datasets")
ultralytics.settings

## Download Dataset

In [ ]:
os.makedirs(HOME/"datasets", exist_ok=True)

%cd {HOME}\datasets

rf = Roboflow(api_key="")
project = rf.workspace("").project("")
version = project.version(1)
dataset = version.download("yolo11")            

print(dataset.location)

%cd {HOME}

## Training

> Run this in cli after running `model.train()`
>
> `tensorboard --logdir runs/`

In [ ]:
RUN_TYPE_NUMBER = "detect"
PRETRAINED_MODEL_NAME = "yolo11n.pt"

In [ ]:
def zero_angle_probability(p: float):
    return (4 * p) / 5

# Load the last saved model
model = YOLO(PRETRAINED_MODEL_NAME)  # or use your custom path

results = model.train(
    data=f"{dataset.location}/data.yaml",
    imgsz=640,
    epochs=500,
    batch=16,
    plots=True,
    save=True,
    save_period=10,
    patience=30,
    cache="disk",
    # lr0=1e-5,
    seed=42,

    # Augmentation
    mosaic=0.0, # Augemntation off so that model detect whole object instead partially
    erasing=0.0,
    fliplr=0.0,
    scale=0.0,
    translate=0.0,
    augmentations=[
        A.RandomRotate90(p=zero_angle_probability(p=0.5))
    ]
    
)


**NOTE:** The results of the completed training are saved in `{HOME}/runs/detect/train/`. Let's examine them.

### Validate fine-tuned model

BEST PT

In [ ]:
import polars as pl

# Load the training log
results = pl.read_csv(f"runs/{RUN_TYPE_NUMBER}/train/results.csv")

# Strip spaces from column names
results = results.rename({
    col: col.strip()
    for col in results.columns
})

# Calculate fitness
results = results.with_columns(
    (
        pl.col("metrics/mAP50(B)") * 0.1
        + pl.col("metrics/mAP50-95(B)") * 0.9
    ).alias("fitness")
)

# Find the epoch with the highest fitness
best_epoch = results["fitness"].arg_max() + 1

print(f"Best model was saved at epoch: {best_epoch}")

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/{RUN_TYPE_NUMBER}/train/confusion_matrix.png')

## Testing

In [ ]:
best_model = YOLO(f"runs/{RUN_TYPE_NUMBER}/train/weights/best.pt")

### Testing on valid

In [ ]:
def load_gt(img_path):
    f = lbl_dir / (Path(img_path).stem + ".txt")
    gts = []
    if f.exists():
        for line in f.read_text().splitlines():
            p = line.split()
            if len(p) < 5:
                continue
            v = np.array(p[1:], dtype=float)
            if len(v) == 4:  # plain box: cx cy w h
                cx, cy, w, h = v
                box = [cx - w/2, cy - h/2, cx + w/2, cy + h/2]
            else:            # polygon -> bounding box
                xs, ys = v[0::2], v[1::2]
                box = [xs.min(), ys.min(), xs.max(), ys.max()]
            gts.append((int(p[0]), box))
    return gts

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else 0

In [ ]:
directory = f"{dataset.location}/valid/images"
lbl_dir = Path(dataset.location) / "valid/labels"
pred_val_dir = "val"


# ---- run predictions and match to ground truth ----
rows, total_gt = [], 0
for r in best_model.predict(source=directory, save=True, project=pred_val_dir, verbose=False, iou=0.1, conf=0.1, stream=True):
    gts = load_gt(r.path)
    total_gt += len(gts)
    used = set()
    preds = sorted(zip(r.boxes.conf.tolist(), r.boxes.cls.tolist(), r.boxes.xyxyn.tolist()),
                   key=lambda x: -x[0])
    for conf, cls, box in preds:
        best, best_i = 0, None
        for i, (gc, gb) in enumerate(gts):
            if i in used or gc != int(cls):
                continue
            s = iou(box, gb)
            if s > best:
                best, best_i = s, i
        tp = best >= 0.5
        if tp:
            used.add(best_i)
        rows.append({"class_name": r.names[int(cls)], "conf": float(conf), "tp": bool(tp)})

Imagine there are 100 real objects in your images. The model makes 90 detections: 80 are correct (TP), 10 are wrong (FP). It missed 20 real objects (FN).

**Precision: "When the model says it found something, how often is it right?"**

precision = TP / (TP + FP) = 80 / 90 = 0.89

High precision means few false alarms. It only looks at what the model *detected*, and ignores the objects it missed.

**Recall: "Of all the real objects, how many did the model find?"**

recall = TP / (TP + FN) = 80 / 100 = 0.80

High recall means few misses. It only looks at the *real objects*, and ignores false alarms.

**F1: "One number that balances precision and recall"**

F1 = 2 × precision × recall / (precision + recall) = 2 × 0.89 × 0.80 / (0.89 + 0.80) ≈ 0.84

It is low if either precision or recall is low. A model can't get a high F1 by being good at only one of them.

**Why they pull against each other**

- **High confidence threshold:** the model only reports detections it is very sure about. Precision goes up, because there are few false alarms. Recall goes down, because it misses the uncertain ones.
- **Low confidence threshold:** the model reports everything. Recall goes up, because it finds more. Precision goes down, because more of the detections are wrong.

That is why your code tests many thresholds and picks the one with the best F1.

**Which one to care about**

- If **missing an object is costly** (e.g. safety, medical), focus on recall.
- If **false alarms are costly** (e.g. every alert needs a human to check), focus on precision.
- If both matter about equally, use F1.

In [ ]:
df = pl.DataFrame(rows, schema={"class_name": pl.Utf8, "conf": pl.Float64, "tp": pl.Boolean})

# ---- precision / recall / F1 per confidence threshold ----
res = []
for t in np.arange(0.1, 0.95, 0.05):
    kept = df.filter(pl.col("conf") >= t)
    tp_n = kept.filter(pl.col("tp")).height
    fp_n = kept.height - tp_n
    p = tp_n / kept.height if kept.height else 0
    rc = tp_n / total_gt if total_gt else 0
    f1 = 2 * p * rc / (p + rc) if (p + rc) else 0
    res.append({"threshold": round(float(t), 2), "TP": tp_n, "FP": fp_n,
                "FN": total_gt - tp_n, "precision": p, "recall": rc, "f1": f1})

table = pl.DataFrame(res)
print("Best threshold by F1:", table.sort("f1", descending=True)["threshold"][0])


best = table.sort("f1", descending=True).row(0, named=True)
total = best["TP"] + best["FN"]

print(f"Total real objects: {total}")
print(f"Best threshold: {best['threshold']}")
print(f"  Found:        {best['TP']} ({best['recall']:.0%})")
print(f"  Missed:       {best['FN']}")
print(f"  False alarms: {best['FP']}")
print(f"  When it detects something, it's right {best['precision']:.0%} of the time")

# how confident is the model when right vs wrong?
print("\nAverage confidence:")
print(f"  Correct detections: {df.filter(pl.col('tp'))['conf'].mean():.2f}")
print(f"  Wrong detections:   {df.filter(~pl.col('tp'))['conf'].mean():.2f}")

# per class
print("\nPer class:")
display(
    df.group_by("class_name")
      .agg(
          pl.len().alias("detections"),
          pl.col("tp").sum().alias("correct"),
          (pl.col("tp").mean() * 100).round(1).alias("pct_correct"),
          pl.col("conf").mean().round(2).alias("avg_conf"),
      )
      .sort("detections", descending=True)
)

# simple table: only a few thresholds
display(table.filter(pl.col("threshold").is_in([0.1, 0.25, 0.5, 0.75, 0.9])))

plt.figure(figsize=(7, 4))
plt.plot(table["threshold"], table["precision"], label="precision")
plt.plot(table["threshold"], table["recall"], label="recall")
plt.plot(table["threshold"], table["f1"], label="F1", linewidth=2)
plt.axvline(best["threshold"], color="gray", linestyle="--", label="best F1")
plt.xlabel("Confidence threshold")
plt.ylabel("Score")
plt.legend()
plt.show()

In [ ]:
# ---------- PART 2: confidence histogram, correct detections only ----------
correct_conf = df.filter(pl.col("tp"))["conf"].to_numpy()

n_bins = 10                                   # change this to get finer/coarser bins
counts, edges = np.histogram(correct_conf, bins=n_bins)   # bins fit the data's min..max automatically

labels = [f"{lo:.2f}-{hi:.2f}" for lo, hi in zip(edges[:-1], edges[1:])]

hist = pl.DataFrame({
    "confidence": labels,
    "correct_detections": counts.tolist(),
    "pct_of_correct": (counts / counts.sum() * 100).round(1).tolist(),
})
display(hist)

plt.figure(figsize=(8, 4))
plt.hist(correct_conf, bins=edges, color="steelblue", edgecolor="white")
plt.xlabel("Confidence")
plt.ylabel("Number of correct detections")
plt.title("Confidence of correctly detected objects")
plt.show()

### Testing on Test Case Recursively

In [ ]:
directory = Path("")

pred_val_dir = "test-outputs"

# Find all images recursively
image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
    ".heic",
    ".heif",
    ".mpo",
    ".dng",
    ".avif",
    ".jp2",
}

images = [
    str(p)
    for p in directory.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
]

print(f"Found {len(images)} images")

test_results = best_model.predict(source=images, save=True, project=pred_val_dir, verbose=False, iou=0.1, conf=0.1, stream=True)

for result in test_results:
    pass